In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
import warnings
warnings.filterwarnings('ignore')

# Read the dataset
df = pd.read_csv('course_lead_scoring.csv')

print("="*70)
print("DATA PREPARATION")
print("="*70)

print(f"\nDataset shape: {df.shape}")
print("\nColumn types:")
print(df.dtypes)

# Check missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Identify categorical and numerical features
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target from numerical features if present
if 'converted' in numerical_features:
    numerical_features.remove('converted')

print(f"\nCategorical features: {categorical_features}")
print(f"Numerical features: {numerical_features}")

# Fill missing values
# Categorical: replace with 'NA'
for col in categorical_features:
    df[col] = df[col].fillna('NA')

# Numerical: replace with 0.0
for col in numerical_features:
    df[col] = df[col].fillna(0.0)

print("\nMissing values after filling:")
print(df.isnull().sum())

# Question 1
print("\n" + "="*70)
print("QUESTION 1: Most Frequent Observation (Mode) for 'industry'")
print("="*70)

industry_mode = df['industry'].mode()[0]
industry_counts = df['industry'].value_counts()
print("\nIndustry value counts:")
print(industry_counts)
print(f"\nMode (most frequent): {industry_mode}")

# Question 2
print("\n" + "="*70)
print("QUESTION 2: Correlation Matrix for Numerical Features")
print("="*70)

# Create correlation matrix for numerical features
numerical_df = df[numerical_features]
correlation_matrix = numerical_df.corr()

print("\nCorrelation matrix:")
print(correlation_matrix)

# Check specific pairs
pairs_to_check = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

print("\nCorrelation coefficients for specified pairs:")
correlations = {}
for feat1, feat2 in pairs_to_check:
    if feat1 in correlation_matrix.columns and feat2 in correlation_matrix.columns:
        corr_value = abs(correlation_matrix.loc[feat1, feat2])
        correlations[f"{feat1} and {feat2}"] = corr_value
        print(f"  {feat1} and {feat2}: {corr_value:.4f}")

max_pair = max(correlations, key=correlations.get)
print(f"\nBiggest correlation: {max_pair} ({correlations[max_pair]:.4f})")

# Split the data
print("\n" + "="*70)
print("SPLITTING THE DATA")
print("="*70)

# Remove target from features
df_features = df.drop('converted', axis=1)
df_target = df['converted']

# First split: 60% train, 40% temp (val+test)
df_train, df_temp, y_train, y_temp = train_test_split(
    df_features, df_target, test_size=0.4, random_state=42
)

# Second split: split the 40% into 20% val and 20% test (50/50 of the temp)
df_val, df_test, y_val, y_test = train_test_split(
    df_temp, y_temp, test_size=0.5, random_state=42
)

print(f"\nTrain set: {len(df_train)} rows ({len(df_train)/len(df)*100:.1f}%)")
print(f"Validation set: {len(df_val)} rows ({len(df_val)/len(df)*100:.1f}%)")
print(f"Test set: {len(df_test)} rows ({len(df_test)/len(df)*100:.1f}%)")

# Question 3
print("\n" + "="*70)
print("QUESTION 3: Mutual Information Score")
print("="*70)

categorical_vars = ['industry', 'location', 'lead_source', 'employment_status']
mi_scores = {}

for var in categorical_vars:
    # Calculate mutual information score
    mi_score = mutual_info_score(df_train[var], y_train)
    mi_scores[var] = round(mi_score, 2)
    print(f"{var}: {round(mi_score, 2)}")

best_mi_var = max(mi_scores, key=mi_scores.get)
print(f"\nHighest mutual information score: {best_mi_var} ({mi_scores[best_mi_var]})")

# Question 4
print("\n" + "="*70)
print("QUESTION 4: Logistic Regression with One-Hot Encoding")
print("="*70)

# Prepare data with one-hot encoding
def prepare_data(df):
    """Convert dataframe to dictionary format for DictVectorizer"""
    return df.to_dict(orient='records')

# Convert to dictionary format
train_dict = prepare_data(df_train)
val_dict = prepare_data(df_val)

# One-hot encoding using DictVectorizer
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

print(f"\nFeature matrix shape after one-hot encoding:")
print(f"Train: {X_train.shape}")
print(f"Validation: {X_val.shape}")

# Train logistic regression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Predict and calculate accuracy
y_pred_val = model.predict(X_val)
accuracy_val = accuracy_score(y_val, y_pred_val)

print(f"\nValidation accuracy: {round(accuracy_val, 2)}")

# Question 5
print("\n" + "="*70)
print("QUESTION 5: Feature Elimination - Least Useful Feature")
print("="*70)

original_accuracy = accuracy_val
print(f"Original accuracy (with all features): {original_accuracy:.4f}")

# Get all feature names
all_features = df_train.columns.tolist()
print(f"\nAll features: {all_features}")

feature_importance = {}

for feature in all_features:
    # Create dataset without this feature
    df_train_reduced = df_train.drop(feature, axis=1)
    df_val_reduced = df_val.drop(feature, axis=1)
    
    # Prepare data
    train_dict_reduced = prepare_data(df_train_reduced)
    val_dict_reduced = prepare_data(df_val_reduced)
    
    # One-hot encoding
    dv_reduced = DictVectorizer(sparse=False)
    X_train_reduced = dv_reduced.fit_transform(train_dict_reduced)
    X_val_reduced = dv_reduced.transform(val_dict_reduced)
    
    # Train model
    model_reduced = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_reduced.fit(X_train_reduced, y_train)
    
    # Calculate accuracy
    y_pred_reduced = model_reduced.predict(X_val_reduced)
    accuracy_reduced = accuracy_score(y_val, y_pred_reduced)
    
    # Calculate difference
    difference = original_accuracy - accuracy_reduced
    feature_importance[feature] = difference
    
    print(f"Without '{feature}': accuracy = {accuracy_reduced:.4f}, difference = {difference:.4f}")

# Find feature with smallest difference
least_useful_feature = min(feature_importance, key=lambda k: abs(feature_importance[k]))
print(f"\nFeature with smallest difference: {least_useful_feature} (difference = {feature_importance[least_useful_feature]:.4f})")

# Question 6
print("\n" + "="*70)
print("QUESTION 6: Regularized Logistic Regression with Different C Values")
print("="*70)

C_values = [0.01, 0.1, 1, 10, 100]
c_accuracies = {}

for C in C_values:
    # Train model with this C value
    model_c = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model_c.fit(X_train, y_train)
    
    # Predict and calculate accuracy
    y_pred_c = model_c.predict(X_val)
    accuracy_c = accuracy_score(y_val, y_pred_c)
    c_accuracies[C] = round(accuracy_c, 3)
    
    print(f"C = {C:6.2f}: accuracy = {round(accuracy_c, 3)}")

# Find best C (smallest C with best accuracy)
best_accuracy = max(c_accuracies.values())
best_C = min([c for c, acc in c_accuracies.items() if acc == best_accuracy])

print(f"\nBest C value: {best_C} (accuracy = {best_accuracy})")

# Summary
print("\n" + "="*70)
print("SUMMARY OF ANSWERS")
print("="*70)
print(f"Q1: Most frequent industry - {industry_mode}")
print(f"Q2: Biggest correlation - {max_pair}")
print(f"Q3: Highest mutual information - {best_mi_var}")
print(f"Q4: Validation accuracy - {round(accuracy_val, 2)}")
print(f"Q5: Least useful feature - {least_useful_feature}")
print(f"Q6: Best C value - {best_C}")

DATA PREPARATION

Dataset shape: (1462, 9)

Column types:
lead_source                  object
industry                     object
number_of_courses_viewed      int64
annual_income               float64
employment_status            object
location                     object
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

Missing values per column:
lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

Categorical features: ['lead_source', 'industry', 'employment_status', 'location']
Numerical features: ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

Missing values after filling:
lead_source                 0
industry                    0